In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 — Imports, Paths, and Global Constants
# ═══════════════════════════════════════════════════════════════════════
!pip install -q timm

import torch, torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
import pandas as pd, numpy as np, json, time, random, itertools
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
import timm
import warnings; warnings.filterwarnings("ignore")

DATA_ROOT = Path("/kaggle/input/datasets/ashery/chexpert")
TRAIN_CSV  = DATA_ROOT / "train.csv"
OUTPUT_DIR = Path("/kaggle/working"); OUTPUT_DIR.mkdir(exist_ok=True)

MODEL_NAME        = "vit_b16"
IMAGE_SIZE        = 224    # ViT-B/16 native resolution (16x16 patches → 14x14=196 patches)
NUM_CLASSES       = 14
DEVICE            = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED              = 42
HP_SEARCH_SUBSET  = 0.05   # 5% for ViT — 86M params needs more GPU memory per trial
HP_SEARCH_EPOCHS  = 5
FINAL_EPOCHS      = 15
HP_PATIENCE       = 3
FINAL_PATIENCE    = 5
MAX_TRIALS        = 36     # Random search (Bergstra & Bengio, 2012)

LABEL_NAMES = [
    "No Finding","Enlarged Cardiomediastinum","Cardiomegaly","Lung Opacity",
    "Lung Lesion","Edema","Consolidation","Pneumonia","Atelectasis",
    "Pneumothorax","Pleural Effusion","Pleural Other","Fracture","Support Devices"
]
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
print(f"Device: {DEVICE}  |  Model: {MODEL_NAME}  |  Image: {IMAGE_SIZE}x{IMAGE_SIZE}")

Device: cuda  |  Model: vit_b16  |  Image: 224x224


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2 — Literature-Justified HP Search Space
# ═══════════════════════════════════════════════════════════════════════
# References:
#  [V1] Ajou Univ CXR ViT — AdamW lr=1e-4..1e-5, batch=32, 15 epochs, imbalanced CXR
#  [V2] Structural MRI ViT — warmup 1ep, lr~5.8e-5, cosine decay, batch=16, sensitive to LR
#  [V3] COVID CXR Fed-ViT  — grad clip norm=1.0 stabilises ViT training on CXR
#  [V4] HuggingFace CheXpert ViT — AdamW lr=3e-5, batch=32, BCEWithLogits, 10 epochs
#  [V5] DR fusion recipe   — AdamW, lr=1e-4, backbone_lr=1e-5, warmup+cosine, clip=5.0
#  NOTE: ViTs are MORE sensitive to LR/scheduler than CNNs — warmup + cosine is essential
HP_SPACE = {
    "learning_rate" : [3e-5, 5e-5, 1e-4],   # [V4] 3e-5; [V2] 5.8e-5; [V1] 1e-4
    "batch_size"    : [16, 24],               # [V2] 16; current code 24; [V1] 32 (memory limited)
    "weight_decay"  : [1e-4, 1e-3],           # [V5] 1e-4; extend to 1e-3 for stronger L2
    "dropout"       : [0.1, 0.3, 0.5],        # [V5] 0.3; explore lower/higher
    "warmup_epochs" : [1, 2],                 # [V2] ~1 epoch warmup; [V3] warmup critical
    "grad_clip"     : [0.5, 1.0, 2.0],        # [V3] 1.0 explicitly recommended; narrow range
}
print("HP Search Space:")
for k,v in HP_SPACE.items(): print(f"  {k:<18}: {v}")
total = 1
for v in HP_SPACE.values(): total *= len(v)
print(f"Total combinations: {total}  |  Random sample: {MAX_TRIALS}")
print(f"NOTE: HP_SEARCH_SUBSET={HP_SEARCH_SUBSET*100:.0f}% (reduced for ViT-86M params)")

HP Search Space:
  learning_rate     : [3e-05, 5e-05, 0.0001]
  batch_size        : [16, 24]
  weight_decay      : [0.0001, 0.001]
  dropout           : [0.1, 0.3, 0.5]
  warmup_epochs     : [1, 2]
  grad_clip         : [0.5, 1.0, 2.0]
Total combinations: 216  |  Random sample: 36
NOTE: HP_SEARCH_SUBSET=5% (reduced for ViT-86M params)


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 — CheXpert Dataset + Transforms [FIXED]
# ═══════════════════════════════════════════════════════════════════════
from PIL import Image
import numpy as np

def csv_to_abs_path(raw_csv_path, data_root):
    rel = raw_csv_path.replace("CheXpert-v1.0-small/", "")
    return Path(data_root) / rel

def filter_existing_files(df, data_root):
    print("  Building valid patient index from disk (fast scan)...")
    train_dir = Path(data_root) / "train"
    valid_dir = Path(data_root) / "valid"
    existing_patients = set()
    for d in [train_dir, valid_dir]:
        if d.exists():
            for p in d.iterdir():
                if p.is_dir():
                    existing_patients.add(p.name)
    def patient_exists(raw_path):
        parts = raw_path.replace("CheXpert-v1.0-small/", "").split("/")
        return parts[1] if len(parts) > 1 else None
    mask = df["Path"].apply(lambda p: patient_exists(p) in existing_patients)
    n_missing = (~mask).sum()
    print(f"  ✓ {mask.sum():,} valid  |  ✗ {n_missing:,} missing (dropped)")
    return df[mask].reset_index(drop=True)

class CheXpertDataset(Dataset):
    def __init__(self, df, image_root, transform=None, uncertain_policy="ones"):
        self.df = df.reset_index(drop=True)
        self.image_root = Path(image_root)
        self.transform = transform
        self.df[LABEL_NAMES] = self.df[LABEL_NAMES].fillna(0)
        self.df[LABEL_NAMES] = self.df[LABEL_NAMES].replace(
            -1, 1 if uncertain_policy == "ones" else 0)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = csv_to_abs_path(row["Path"], self.image_root)
        img      = Image.open(img_path).convert("RGB")
        lbl      = torch.tensor(row[LABEL_NAMES].values.astype(np.float32))
        if self.transform: img = self.transform(img)
        return img, lbl

def get_transforms(train=True):
    if train:
        return T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                          T.RandomHorizontalFlip(0.5), T.RandomRotation(15),
                          T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
                          T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
                          T.ToTensor(),
                          T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    return T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)), T.ToTensor(),
                      T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

# ── Load, filter, split ───────────────────────────────────────────────
print("Loading CheXpert data...")
full_df = pd.read_csv(TRAIN_CSV)
print(f"  Raw CSV rows: {len(full_df):,}")

full_df = filter_existing_files(full_df, DATA_ROOT)

hp_df, _         = train_test_split(full_df, test_size=1-HP_SEARCH_SUBSET, random_state=SEED)
hp_train, hp_val = train_test_split(hp_df,   test_size=0.15,               random_state=SEED)
print(f"  HP train: {len(hp_train):,}  |  HP val: {len(hp_val):,}  |  Full (clean): {len(full_df):,}")

Loading CheXpert data...
  Raw CSV rows: 223,414
  Building valid patient index from disk (fast scan)...
  ✓ 223,414 valid  |  ✗ 0 missing (dropped)
  HP train: 9,494  |  HP val: 1,676  |  Full (clean): 223,414


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4 — ViT-B/16 Model + Optimizer Builder
# ═══════════════════════════════════════════════════════════════════════
class CheXpertViT(nn.Module):
    """ViT-B/16 with custom classifier head — tunable dropout."""
    def __init__(self, hp, pretrained=True):
        super().__init__()
        self.backbone   = timm.create_model("vit_base_patch16_224", pretrained=pretrained, num_classes=0)
        embed_dim       = self.backbone.embed_dim  # 768 for ViT-Base
        drop            = float(hp["dropout"])
        self.classifier = nn.Sequential(nn.LayerNorm(embed_dim), nn.Dropout(drop),
                                        nn.Linear(embed_dim, NUM_CLASSES))
    def forward(self, x):
        return self.classifier(self.backbone(x))

def build_model(hp):
    return CheXpertViT(hp, pretrained=True)

def build_opt_sched(model, hp, total_epochs=HP_SEARCH_EPOCHS):
    wu = int(hp["warmup_epochs"])
    opt = AdamW(model.parameters(), lr=float(hp["learning_rate"]), weight_decay=float(hp["weight_decay"]))
    # Warmup + cosine decay — critical for ViT stability (Dosovitskiy et al., 2020)
    def lr_lambda(epoch):
        if epoch < wu:   return (epoch + 1) / wu           # linear warmup
        return 0.5 * (1 + np.cos(np.pi * (epoch - wu) / max(total_epochs - wu, 1)))
    sch = LambdaLR(opt, lr_lambda=lr_lambda)
    return opt, sch

In [5]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5 — Class Weight Computation
# ═══════════════════════════════════════════════════════════════════════
def compute_pos_weights(loader):
    pos, total = torch.zeros(NUM_CLASSES), 0
    for _, lbl in loader:
        pos += lbl.sum(0); total += lbl.size(0)
    return (total - pos) / (pos + 1e-5)

In [6]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — Train / Validate Functions (with Gradient Clipping for ViT)
# ═══════════════════════════════════════════════════════════════════════
def train_one_epoch(model, loader, criterion, optimizer, scaler, grad_clip):
    model.train(); total_loss = 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            loss = criterion(model(imgs), lbls)
        scaler.scale(loss).backward()
        # Gradient clipping — essential for ViT: prevents exploding gradients [V3]
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)

def validate_epoch(model, loader, criterion):
    model.eval(); total_loss, preds, labels = 0, [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            with torch.amp.autocast("cuda"):
                logits = model(imgs); loss = criterion(logits, lbls)
            total_loss += loss.item()
            preds.append(torch.sigmoid(logits).cpu().numpy())
            labels.append(lbls.cpu().numpy())
    p = np.concatenate(preds); l = np.concatenate(labels)
    aurocs = [roc_auc_score(l[:,i],p[:,i]) for i in range(NUM_CLASSES) if l[:,i].sum()>0]
    return total_loss/len(loader), np.mean(aurocs), p, l

In [7]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 7 — HP Search Loop
# ═══════════════════════════════════════════════════════════════════════
keys   = list(HP_SPACE.keys())
combos = [dict(zip(keys,v)) for v in itertools.product(*HP_SPACE.values())]
random.shuffle(combos); combos = combos[:MAX_TRIALS]
print(f"\n{'='*68}\n  PHASE 1 — HP SEARCH  |  {MODEL_NAME.upper()}  |  {len(combos)} trials\n{'='*68}")

all_results = []
for t_idx, hp in enumerate(combos, 1):
    print(f"\n  Trial {t_idx:>2}/{len(combos)} | lr={hp['learning_rate']} | bs={hp['batch_size']} | wd={hp['weight_decay']} | drop={hp['dropout']} | wu={hp['warmup_epochs']} | clip={hp['grad_clip']}")
    bs      = int(hp["batch_size"])
    t_ds    = CheXpertDataset(hp_train, DATA_ROOT, get_transforms(True))
    v_ds    = CheXpertDataset(hp_val,   DATA_ROOT, get_transforms(False))
    t_ld    = DataLoader(t_ds, bs, shuffle=True,  num_workers=4, pin_memory=True)
    v_ld    = DataLoader(v_ds, bs, shuffle=False, num_workers=4, pin_memory=True)
    model   = build_model(hp).to(DEVICE)
    pos_w   = compute_pos_weights(t_ld).to(DEVICE)
    crit    = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt,sch = build_opt_sched(model, hp)
    scaler  = torch.amp.GradScaler("cuda")
    epoch_logs, best_auroc, patience = [], 0.0, 0
    t_start = time.time()
    for ep in range(1, HP_SEARCH_EPOCHS+1):
        tr_loss             = train_one_epoch(model, t_ld, crit, opt, scaler, float(hp["grad_clip"]))
        vl_loss, auroc, _,_ = validate_epoch(model, v_ld, crit)
        cur_lr = opt.param_groups[0]["lr"]
        if sch: sch.step()
        warmup_tag = " [WARMUP]" if ep <= int(hp["warmup_epochs"]) else ""
        epoch_logs.append({"trial":t_idx,"epoch":ep,"train_loss":round(tr_loss,5),
                           "val_loss":round(vl_loss,5),"val_auroc":round(auroc,5),"lr":round(cur_lr,8)})
        print(f"    Ep{ep} | TrLoss:{tr_loss:.4f} VlLoss:{vl_loss:.4f} AUROC:{auroc:.4f} LR:{cur_lr:.2e}{warmup_tag}")
        if auroc > best_auroc: best_auroc=auroc; patience=0
        else:
            patience+=1
            if patience>=HP_PATIENCE: print(f"    ⚠ Early stop"); break
    elapsed = round(time.time()-t_start,1)
    result  = {"trial":t_idx,"model":MODEL_NAME,"best_auroc":round(best_auroc,5),
               "time_s":elapsed,"epoch_log":epoch_logs,**{f"hp_{k}":v for k,v in hp.items()}}
    all_results.append(result)
    with open(OUTPUT_DIR/f"hp_log_{MODEL_NAME}.json","w") as f: json.dump(all_results,f,indent=2)
    print(f"  ✓ Best AUROC: {best_auroc:.4f}  |  {elapsed}s")


  PHASE 1 — HP SEARCH  |  VIT_B16  |  36 trials

  Trial  1/36 | lr=3e-05 | bs=24 | wd=0.0001 | drop=0.1 | wu=1 | clip=0.5


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

    Ep1 | TrLoss:1.0723 VlLoss:1.0321 AUROC:0.6798 LR:3.00e-05 [WARMUP]
    Ep2 | TrLoss:1.0224 VlLoss:0.9806 AUROC:0.7132 LR:3.00e-05
    Ep3 | TrLoss:0.9828 VlLoss:0.9604 AUROC:0.7282 LR:2.56e-05
    Ep4 | TrLoss:0.9382 VlLoss:0.9460 AUROC:0.7409 LR:1.50e-05
    Ep5 | TrLoss:0.8816 VlLoss:0.9261 AUROC:0.7456 LR:4.39e-06
  ✓ Best AUROC: 0.7456  |  874.6s

  Trial  2/36 | lr=0.0001 | bs=16 | wd=0.0001 | drop=0.1 | wu=1 | clip=2.0
    Ep1 | TrLoss:1.1218 VlLoss:1.0964 AUROC:0.6050 LR:1.00e-04 [WARMUP]
    Ep2 | TrLoss:1.1007 VlLoss:1.0715 AUROC:0.6100 LR:1.00e-04
    Ep3 | TrLoss:1.0804 VlLoss:1.0441 AUROC:0.6318 LR:8.54e-05
    Ep4 | TrLoss:1.0636 VlLoss:1.0349 AUROC:0.6382 LR:5.00e-05
    Ep5 | TrLoss:1.0502 VlLoss:1.0259 AUROC:0.6446 LR:1.46e-05
  ✓ Best AUROC: 0.6446  |  924.0s

  Trial  3/36 | lr=3e-05 | bs=16 | wd=0.001 | drop=0.5 | wu=1 | clip=0.5
    Ep1 | TrLoss:1.1649 VlLoss:1.0949 AUROC:0.6297 LR:3.00e-05 [WARMUP]
    Ep2 | TrLoss:1.1067 VlLoss:1.0294 AUROC:0.6621 LR:3.00e-05

In [8]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 8 — Results Analysis + Optimal Config
# ═══════════════════════════════════════════════════════════════════════
rows = [{"trial":r["trial"],"best_auroc":r["best_auroc"],"time_s":r["time_s"],
         **{k.replace("hp_",""):v for k,v in r.items() if k.startswith("hp_")}}
        for r in all_results]
summary = pd.DataFrame(rows).sort_values("best_auroc", ascending=False)
summary.to_csv(OUTPUT_DIR/f"hp_summary_{MODEL_NAME}.csv", index=False)
print(f"\nTOP-10 TRIALS:\n{summary.head(10).to_string(index=False)}")

hp_cols = [c for c in summary.columns if c not in ["trial","best_auroc","time_s"]]
sens_lines = ["SENSITIVITY ANALYSIS"]
for col in hp_cols:
    grp = summary.groupby(col)["best_auroc"].mean().sort_values(ascending=False)
    sens_lines.append(f"  {col}:")
    for val,auroc in grp.items():
        bar = "█"*max(1,int((auroc-grp.min())/(grp.max()-grp.min()+1e-9)*20))
        sens_lines.append(f"    {str(val):>10}  {bar:<22} mean AUROC={auroc:.4f}")
sens_text = "\n".join(sens_lines)
print(f"\n{sens_text}")
with open(OUTPUT_DIR/f"sensitivity_{MODEL_NAME}.txt","w") as f: f.write(sens_text)

best_row = summary.iloc[0]
optimal  = {col:best_row[col] for col in hp_cols}
optimal["val_auroc_phase1"] = best_row["best_auroc"]
with open(OUTPUT_DIR/f"optimal_{MODEL_NAME}.json","w") as f: json.dump(optimal,f,indent=2)
print(f"\nOPTIMAL CONFIG: {optimal}")


TOP-10 TRIALS:
 trial  best_auroc  time_s  learning_rate  batch_size  weight_decay  dropout  warmup_epochs  grad_clip
    18     0.74909   925.5        0.00003          16        0.0001      0.1              2        2.0
    32     0.74775   924.8        0.00003          16        0.0001      0.1              2        0.5
    35     0.74720   876.1        0.00003          24        0.0001      0.3              1        2.0
     1     0.74559   874.6        0.00003          24        0.0001      0.1              1        0.5
    28     0.74450   927.9        0.00003          16        0.0001      0.1              1        2.0
    21     0.74372   925.6        0.00003          16        0.0010      0.1              2        0.5
    26     0.74245   880.1        0.00003          24        0.0001      0.3              1        0.5
     6     0.74233   879.1        0.00003          24        0.0001      0.3              2        0.5
    33     0.73981   924.7        0.00003          16    